In [3]:
#STEP 0 - create the spark session. Always run this cell first (after every kernel restart).
#(inside a Databricks workspace notebook `spark` already exists; locally in VS Code we must create it)
import glob, os, shutil
from databricks.connect import DatabricksSession

#auth uses the Databricks CLI login; if the CLI is not on PATH, point to the copy bundled with the VS Code Databricks extension
if not shutil.which("databricks") and not os.environ.get("DATABRICKS_CLI_PATH"):
    bundled = sorted(glob.glob(os.path.expanduser("~/.vscode/extensions/databricks.databricks-*/bin/databricks")))
    if bundled:
        os.environ["DATABRICKS_CLI_PATH"] = bundled[-1]

#serverless(True) tells Databricks Connect to run on serverless compute (otherwise it asks for a cluster id)
spark = DatabricksSession.builder.serverless(True).getOrCreate()
print("spark session ready")

spark session ready


In [4]:
#we will use a dataframe created from a list of python tuples to demonstrate.
#first we will define the schema and then create a dataframe from the list of tuples.
schema = "id int, name string, age short, salary double"
data_list = [(100, "John", 30, 50000.0), 
(101, "Jane", 25, 60000.0), 
(102, "Bob", 35, 70000.0)]
#then we will create a dataframe from the list of tuples and the schema defined above.
sample_df = spark.createDataFrame(data=data_list, schema=schema)

sample_df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: short (nullable = true)
 |-- salary: double (nullable = true)



In [5]:
#Adding the following columns to the Dataframe
#1. increment: 10% of the salary up to 3000 maximum increment
#2. revised_salary: salary + increment

#withColumn adds a new column, or replaces an existing column if you reuse its name (e.g. to transform it or cast its type).
#withColumns does the same for several columns at once, taking a dict of {column_name: expression}.
#It can NOT rename or drop columns: use withColumnRenamed("old", "new") to rename and drop("col") to drop.

#Note: all expressions inside one withColumns call are evaluated against the ORIGINAL dataframe,
#so revised_salary can't refer to increment here; we repeat the increment formula instead.

from pyspark.sql.functions import expr

salary_df = (
    sample_df.withColumns({
        "increment": expr("least(salary * 0.1, 3000)"),
        "revised_salary": expr("salary + least(salary * 0.1, 3000)")
    })
)

salary_df.show()

+---+----+---+-------+---------+--------------+
| id|name|age| salary|increment|revised_salary|
+---+----+---+-------+---------+--------------+
|100|John| 30|50000.0|   3000.0|       53000.0|
|101|Jane| 25|60000.0|   3000.0|       63000.0|
|102| Bob| 35|70000.0|   3000.0|       73000.0|
+---+----+---+-------+---------+--------------+



In [6]:
#but if we wanted to add increment: 10% of the salary up to 3000 maximum increment,
#then update the salary right away instead of adding another column called revised salary,
#we will not be able to do it with the same way, because Spark can’t find a column called increment, because when the salary line is calculated, increment doesn’t exist yet,
# Why: withColumns does everything at the same time,
# Both lines look at the same starting table. increment only appears after withColumns finishes, so the second line can’t use it.
# So the Fix is to Do it in two steps:

from pyspark.sql.functions import expr

salary_df = (
    sample_df.withColumn("increment", expr("least(salary * 0.1, 3000)"))
        .withColumn("salary", expr("salary + increment"))
)

salary_df.show()

+---+----+---+-------+---------+
| id|name|age| salary|increment|
+---+----+---+-------+---------+
|100|John| 30|53000.0|   3000.0|
|101|Jane| 25|63000.0|   3000.0|
|102| Bob| 35|73000.0|   3000.0|
+---+----+---+-------+---------+



In [7]:
# ####4. Add a batch number (uuid) column to dataframe

#generate a UUID using python then convert it to string because spark doesn't support uuid data type

#Import the tools
import uuid
from pyspark.sql.functions import lit
#Create one unique ID for this run
batch_id = str(uuid.uuid4())
#Stamp that ID on every row, Same ID on all rows, because they all came in the same batch.
salary_batch_df = sample_df.withColumn("batch_id", lit(batch_id))

salary_batch_df.show(truncate=False)

+---+----+---+-------+------------------------------------+
|id |name|age|salary |batch_id                            |
+---+----+---+-------+------------------------------------+
|100|John|30 |50000.0|22daacd7-dc74-4d36-9299-3e9397817c78|
|101|Jane|25 |60000.0|22daacd7-dc74-4d36-9299-3e9397817c78|
|102|Bob |35 |70000.0|22daacd7-dc74-4d36-9299-3e9397817c78|
+---+----+---+-------+------------------------------------+



In [10]:
#Rename the dataframe colums increment as annual_increment,
# & salary as incremented_salary

'''withcolumnsrename is used to rename a column in a dataframe. 
it takes two arguments: the old column name and the new column name.'''

new_salary_df = (
    salary_df.withColumnsRenamed({
        "increment": "annual_increment",
        "salary": "incremented_salary"
    })
)

display(new_salary_df)

,id,name,age,incremented_salary,annual_increment
0,100,John,30,53000.0,3000.0
1,101,Jane,25,63000.0,3000.0
2,102,Bob,35,73000.0,3000.0


In [11]:
# Remove the age, annual_increment colums from the dataframe

removed_df = new_salary_df.drop("age", "annual_increment")

display(removed_df)

,id,name,incremented_salary
0,100,John,53000.0
1,101,Jane,63000.0
2,102,Bob,73000.0
